# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook demonstrates how to explore and process the FAIR\(\textsuperscript{2}\) dataset “Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells” using the [mlcroissant](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL (see next cell).

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load dataset metadata and examine the main description using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Print a short description
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

## 2. Data Overview

Review available record sets, fields, and their IDs defined in the Croissant metadata.

In [ ]:
# List all record set @id's and their fields
print("Available record sets:")
for record_set in dataset.metadata.record_sets:
    print(f"- {record_set['@id']}: {record_set['name']}")
    if 'fields' in record_set:
        for field in record_set['fields']:
            print(f"    - field @id={field['@id']} name='{field.get('name')}'")

## 3. Data Extraction

Load data from all record sets into DataFrames for exploration and analysis. All dataset entities used below are referred to by their Croissant `@id` fields.

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

# Discover all record sets
record_sets = [rs['@id'] for rs in dataset.metadata.record_sets]
print(f"All record set IDs: {record_sets}")
dataframes = {}

# Load each record set as DataFrame (using @id)
for record_set_id in record_sets:
    try:
        records = list(dataset.records(record_set=record_set_id))
        if len(records) > 0:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Loaded {len(df)} records from record set {record_set_id}. Fields: {df.columns.tolist()}")
        else:
            print(f"Record set {record_set_id} is empty.")
    except Exception as e:
        print(f"Failed loading record set {record_set_id}: {str(e)}")

# For demonstration, look at the first non-empty record set
main_record_set_id = None
for rsid in record_sets:
    if rsid in dataframes and not dataframes[rsid].empty:
        main_record_set_id = rsid
        break
if main_record_set_id:
    print(f"\nUsing main record set: {main_record_set_id}")
    print(f"Fields: {dataframes[main_record_set_id].columns.tolist()}")
    display(dataframes[main_record_set_id].head())
else:
    print("No non-empty record sets found.")

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps such as filtering records, normalizing numeric fields, and grouping. All operations use Croissant entity `@id`'s for reference.

In [ ]:
# EDA on the primary record set
import numpy as np

df = dataframes.get(main_record_set_id)
if df is not None:
    print(f"Available fields in main record set ({main_record_set_id}): {df.columns.tolist()}")
    # Example: Let's heuristically choose 'cr:MW' (molecular weight) as a numeric analysis field if present
    numeric_field_id = None
    for col in df.columns:
        if 'MW' in col or 'mass' in col or col == 'cr:MW':
            numeric_field_id = col
            break
    if numeric_field_id is None:
        # Default to first number-like column
        for col in df.columns:
            if np.issubdtype(df[col].dtype, np.number):
                numeric_field_id = col
                break
    if numeric_field_id is not None:
        try:
            df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
        except Exception:
            pass
        numeric_vals = df[numeric_field_id].dropna()
        threshold = numeric_vals.mean() if not numeric_vals.empty else 0
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"\nFiltered records with {numeric_field_id} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalization
        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / (filtered_df[numeric_field_id].std() + 1e-8)
        print(f"\nNormalized '{numeric_field_id}' for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Grouping by a likely categorical field
        group_field_id = None
        for cat_candidate in ['cr:description', 'cr:accession', 'cr:sample', 'cr:group']:
            if cat_candidate in df.columns:
                group_field_id = cat_candidate
                break
        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
            print(f"\nGrouped by {group_field_id}, mean of {numeric_field_id}:")
            display(grouped_df.head())
    else:
        print("No numeric field found for analysis.")
else:
    print("No main record set available.")

## 5. Visualization

Visualize the distribution of a selected numeric field and, optionally, compare categorical groupings.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if df is not None and numeric_field_id is not None:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # If group_field_id exists, boxplot by group
    if group_field_id:
        top_groups = df[group_field_id].value_counts().nlargest(10).index
        plt.figure(figsize=(12, 6))
        sns.boxplot(data=df[df[group_field_id].isin(top_groups)], x=group_field_id, y=numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id} (top 10 groups)")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion

We have loaded the FAIR$^2$ dataset using its Croissant schema, explored metadata and record set structure with all Croissant references by `@id`, extracted records into DataFrames, filtered and normalized a numeric field, grouped by a categorical value, and visualized record distributions. 

For further analysis, consult the dataset documentation and metadata for the specific meaning of each field and utilize the `mlcroissant` library for scalable FAIR data processing.